# Integrated Local 3D Reconstruction

这个 notebook 只保留本地可运行的重建部分，抛开 ROS / 扫描硬件控制。

包含两条流程：
1. `RGB-D stitching`：更接近原始 notebook 的逐帧拼接。
2. `Canopy reconstruction`：更适合这批顶视植物数据的掩膜 + 深度冠层重建。

> 说明：第二条流程重建的是植物可见冠层表面，属于更稳定的 `2.5D` 植物表面，而不是完整 360 度植物体积模型。

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "processing").exists() and (candidate / "main.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the PhenoFusion3D repo root from the current working directory.")


PROJECT_ROOT = find_repo_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"PYTHONPATH updated = {PROJECT_ROOT in map(Path, [Path(p) for p in sys.path if p])}")

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import open3d as o3d
from IPython.display import Image, display

from processing.canopy import CanopyReconstructionConfig, reconstruct_canopy
from processing.reconstructor import ReconstructionConfig, reconstruct_sequence


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def show_image(path: Path, width: int | None = None):
    path = Path(path)
    if not path.exists():
        print(f"Missing image: {path}")
        return
    display(Image(filename=str(path), width=width))


def preview_point_cloud(path: Path, title: str, sample_limit: int = 80000):
    path = Path(path)
    if not path.exists():
        print(f"Missing point cloud: {path}")
        return

    pcd = o3d.io.read_point_cloud(str(path))
    points = np.asarray(pcd.points)
    if len(points) == 0:
        print(f"Empty point cloud: {path}")
        return

    if pcd.has_colors():
        colors = np.asarray(pcd.colors)
    else:
        colors = np.repeat(np.array([[0.2, 0.6, 0.2]], dtype=np.float32), len(points), axis=0)

    step = max(1, len(points) // sample_limit)
    points = points[::step]
    colors = colors[::step]

    fig = plt.figure(figsize=(12, 5), dpi=140)
    ax1 = fig.add_subplot(1, 2, 1)
    ax1.scatter(points[:, 0], points[:, 1], s=0.4, c=colors)
    ax1.set_title(f"{title} - Top View")
    ax1.set_aspect("equal")
    ax1.axis("off")

    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    ax2.scatter(points[:, 0], points[:, 1], points[:, 2], s=0.4, c=colors, depthshade=False)
    ax2.view_init(elev=28, azim=-55)
    ax2.set_title(f"{title} - Oblique")
    ax2.set_axis_off()

    mins = points.min(axis=0)
    maxs = points.max(axis=0)
    center = (mins + maxs) / 2.0
    span = float((maxs - mins).max() / 2.0)
    ax2.set_xlim(center[0] - span, center[0] + span)
    ax2.set_ylim(center[1] - span, center[1] + span)
    ax2.set_zlim(center[2] - span * 0.25, center[2] + span * 0.75)

    fig.tight_layout()
    plt.show()
    plt.close(fig)

## Parameters

下面这组参数默认会把输出写到 `test_plant_rs13_1/notebook_*` 目录里，避免覆盖其他结果。

In [ ]:
DATASET = PROJECT_ROOT / "test_plant_rs13_1"
RGBD_OUTPUT = DATASET / "notebook_rgbd_local"
CANOPY_OUTPUT = DATASET / "notebook_canopy_local"

RUN_RGBD = True
RUN_CANOPY = True

RGBD_CONFIG = ReconstructionConfig(
    step_size=24,
    start_index=220,
    end_index=460,
    depth_scale=1000.0,
    depth_trunc=None,
    voxel_size=0.005,
    min_fitness=0.01,
    output_dir=str(RGBD_OUTPUT),
    green_only=True,
)

CANOPY_CONFIG = CanopyReconstructionConfig(
    max_frames=9,
    min_mask_area=180000,
    coverage_threshold=2,
    z_scale=1.0,
    output_dir=str(CANOPY_OUTPUT),
)

print(f"DATASET      = {DATASET}")
print(f"RGBD_OUTPUT  = {RGBD_OUTPUT}")
print(f"CANOPY_OUTPUT= {CANOPY_OUTPUT}")

## 1. RGB-D Stitching

这一段更接近原始 `3D_reconstruction.ipynb` 的思路：把若干 RGB-D 帧投成点云，再用 ICP 逐帧拼接。对于这批数据，它能给出一个可运行的本地重建，但植物形状通常不如下一节稳定。

In [ ]:
rgbd_summary_path = RGBD_OUTPUT / "reconstruction_summary.json"
if RUN_RGBD or not rgbd_summary_path.exists():
    rgbd_result = reconstruct_sequence(DATASET, config=RGBD_CONFIG)

rgbd_summary = load_json(rgbd_summary_path)
{
    "frames_total": rgbd_summary["frames_total"],
    "frames_selected": rgbd_summary["frames_selected"],
    "frames_registered": rgbd_summary["frames_registered"],
    "frames_failed": rgbd_summary["frames_failed"],
    "final_point_count": rgbd_summary["final_point_count"],
    "depth_trunc_used": rgbd_summary["depth_trunc_used"],
    "output_dir": rgbd_summary["output_dir"],
}

In [ ]:
preview_point_cloud(RGBD_OUTPUT / "merge_pcd_cam.ply", title="RGB-D Stitching Result")

## 2. Plant Canopy Reconstruction

这一段是为当前这批顶视植物数据新增的整合版流程：
- 使用 `reconstruction/masks/mask_*.png` 选出植物帧
- 对齐最完整的几帧植物视图
- 在掩膜内补全深度
- 输出植物冠层点云与网格

这条流程更容易得到“像植物”的结果。

In [ ]:
canopy_summary_path = CANOPY_OUTPUT / "canopy_summary.json"
if RUN_CANOPY or not canopy_summary_path.exists():
    canopy_result = reconstruct_canopy(DATASET, config=CANOPY_CONFIG)

canopy_summary = load_json(canopy_summary_path)
{
    "frames_available": canopy_summary["frames_available"],
    "frames_used": canopy_summary["frames_used"],
    "reference_token": canopy_summary["reference_token"],
    "reference_depth_m": canopy_summary["reference_depth_m"],
    "final_point_count": canopy_summary["final_point_count"],
    "final_triangle_count": canopy_summary["final_triangle_count"],
    "tokens_used": [frame["token"] for frame in canopy_summary["frames"]],
}

In [ ]:
show_image(CANOPY_OUTPUT / "fused_rgb_masked.png", width=500)
show_image(CANOPY_OUTPUT / "canopy_topdown.png", width=950)
show_image(CANOPY_OUTPUT / "canopy_oblique.png", width=700)

In [ ]:
preview_point_cloud(CANOPY_OUTPUT / "canopy_points.ply", title="Canopy Reconstruction Result")

## Notes

- `RGB-D stitching` 更接近原始 notebook 的重建逻辑，但对这批顶视轨道扫描数据更容易出现漂移和非植物结构。
- `Canopy reconstruction` 更适合当前数据，结果更像植物，但它重建的是可见冠层表面，不是完整植物体积。
- 即使有深度图，深度相机也只能记录“当前视线看到的最前表面”，看不到被上层叶片挡住的叶背和茎。

对应 CLI 命令：

```bash
python main.py --input test_plant_rs13_1 --step-size 24 --start-index 220 --end-index 460 --green-filter --output-dir test_plant_rs13_1/notebook_rgbd_local
python main.py --mode canopy --input test_plant_rs13_1 --output-dir test_plant_rs13_1/notebook_canopy_local
```